# GeoSPARQL Extension Functions (rdflib-geosparql)

The following functions are implemented in rdflib-geosparql, but are not (yet) officially standardized by OGC.

## Installing requirements

In [1]:
!pip install -r requirements.txt --break-system-packages
!pip install ipyleaflet --break-system-packages
!jupyter labextension install jupyter-leaflet

import rdflib
import json
import shapely.geometry
from shapely import wkt
from shapely.ops import transform
from rdflib import Graph, Literal
import os
import itertools
from shapely.testing import assert_geometries_equal
from geosparql.geosparql import LiteralUtils
from geosparql.geosparql_aggregates import processLiteralTypeToGeom
from IPython.display import display, HTML
import ipywidgets as W
from ipyleaflet import Map, WKTLayer, GeoJSON, FullScreenControl, LayersControl, Popup#, Tooltip

mapcenter=(34.1,-83.2)
zoomlevel=10
GEO = "http://www.opengis.net/ont/geosparql#"
GEOF = "http://www.opengis.net/def/function/geosparql/"
GEOFEXT = "http://www.opengis.net/def/function/geosparql/ext/"

def getHTMLStringFromQueryResult(qres):
    res="<table><tr><th>Variable</th><th>Value</th></tr>"
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in resultlist:
        for item in row:
            if isinstance(row[item],Literal) and row[item].datatype!=None:
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item].datatype)+"\">"+str(row[item]).strip()+"</a></td></tr>"
            elif isinstance(row[item],URIRef):
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item])+"\">"+str(row[item]).strip()+"</a></td></tr>"
            else:
                res+="<tr><td>"+str(item)+"</td><td>"+str(row[item]).strip()+"</td></tr>"
    res+="</table>"
    return res

def getMapFromWKTResult(qres,rows=[],lmap=None):
    layers=[]
    lastcentroid=shapely.geometry.Point(mapcenter[0],mapcenter[1])
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in rows:
        if row in resultlist[0]:
            popup="<h3>"+str(row)+"</h3><ul>"
            for rowres in resultlist:
                for item in rowres:
                    if isinstance(rowres[item],Literal) and rowres[item].datatype!=None:
                        popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item].datatype)+"\">"+str(rowres[item]).strip()+"</a></li>"
                    elif isinstance(rowres[item],URIRef):
                        popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item])+"\">"+str(rowres[item]).strip()+"</a></li>"
                    else:
                        popup+="<li><b>"+str(item)+":</b> "+str(rowres[item]).strip()+"</li>"
            popup+="</ul>"
            toprocess=resultlist[0][row].strip()
            if toprocess.startswith("<http"):
                toprocess=toprocess[toprocess.find(">")+1:]
            geom = wkt.loads(toprocess)
            if not shapely.is_empty(geom):
                lastcentroid=geom.centroid
                nlayer=WKTLayer(name=str(row),wkt_string=geom.wkt,hover_style={"fillColor": "red"})
                msg=W.HTML()
                msg.value=popup
                nlpopup=Popup(
                    location=[lastcentroid.y,lastcentroid.x],
                    child=msg,
                    close_button=True,
                    auto_close=False,
                    close_on_escape_key=False
                )
                nlayer.popup=msg
                layers.append(nlayer)
    if lmap==None:
        lmap=Map(center=(lastcentroid.y,lastcentroid.x), zoom=zoomlevel)
        control = FullScreenControl()
        lmap.add(control)
        control = LayersControl(position='topright')
        lmap.add(control)
    for lay in layers:
        lmap.add(lay)
    return lmap    

def processQueryResults(qres,rows=[],lmap=None):
    display(HTML(getHTMLStringFromQueryResult(qres)))
    if rows!=[]:
        lmap=getMapFromWKTResult(qres,rows,lmap)
    return lmap

g=Graph()
g.parse("tests/testdata.ttl")
print(len(g))

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:54: UserWarning: An error occurred.
  warnings.warn("An error occurred.")
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:55: UserWarning: PermissionError: [Errno 13] Permission denied: '/usr/local/share/jupyter/lab/extensions/jupyter-leaflet-0.20.0.tgz'
  warnings.warn(msg[-1].strip())
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:56: UserWarning: See the log file for details: /tmp/jupyterlab-debug-lkzbt2wi.log


## GeoSPARQL Serializations

### geofext:asGeoCode Function

In [2]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gc
WHERE {
  my:A geo:hasGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGeocode(?aLiteral,"http://opengis.net/ont/geocode/OpenLocationCode") as ?gc)
}
"""),["aLiteral"],None)

POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
http://opengis.net/ont/geocode/OpenLocationCode


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gc,2G8PJ822+22


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asGLTF Function

In [3]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gltf
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGLTF(?aLiteral) as ?gltf)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gltf,


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asOBJ Function

In [4]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?obj
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asOBJ(?aLiteral) as ?obj)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asPLY Function

In [5]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?ply
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asPLY(?aLiteral) as ?ply)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
ply,


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asJSONFG Function

In [6]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?jsonfg
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asJSONFG(?aLiteral) as ?jsonfg)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asSVG Function

In [7]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?svg
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asSVG(?aLiteral) as ?svg)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
svg,


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Directional relation functions

### geofext:above Function

In [8]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?abovee) as ?above) (xsd:boolean(?nabovee) as ?notabove)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:above(?aLiteral, ?dLiteral) as ?nabovee)
  BIND (geof:above(?dLiteral, ?aLiteral) as ?abovee)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"
above,true
notabove,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:below Function

In [9]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?beloww) as ?D_Below_A) (xsd:boolean(?nbeloww) as ?A_Below_D)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:below(?aLiteral, ?dLiteral) as ?nbeloww)
  BIND (geof:below(?dLiteral, ?aLiteral) as ?beloww)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"
D_Below_A,true
A_Below_D,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:leftOf Function

Checks if the first input geometry is left of the second input geometry. Geometry coordinate reference systems are normalized if they differ.

In [10]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?leftt) as ?A_leftOf_D) (xsd:boolean(?nleftt) as ?D_LeftOf_A)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:leftOf(?aLiteral, ?dLiteral) as ?leftt)
  BIND (geof:leftOf(?dLiteral, ?aLiteral) as ?nleftt)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"
A_leftOf_D,true
D_LeftOf_A,false


Map(center=[34.099999999999994, -82.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

### geofext:rightOf Function

In [11]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?rightt) as ?right) (xsd:boolean(?nrightt) as ?notright)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:rightOf(?aLiteral, ?dLiteral) as ?nrightt)
  BIND (geof:rightOf(?dLiteral, ?aLiteral) as ?rightt)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"
right,true
notright,false


Map(center=[34.099999999999994, -82.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

## Geometry Accessor Functions

### geofext:endPoint Function

Returns the last point of a given geometry in the CRS and literal format of the input geometry.

In [12]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?endPoint
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:endPoint(?literal) AS ?endPoint)
}
"""),["literal","endPoint"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
endPoint,POINT (-83.6 34.1)


Map(center=[34.1, -83.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:M Function

Returns the measurement coordinate of a point geometry.

In [13]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?m
WHERE {
  my:PPointGeom geo:asWKT ?literal .
  BIND(geof:M(?literal) AS ?m)
}
"""),["literal"],None)

Variable,Value
literal,Point M (-83.4 34.0 5.0)
m,5.0


Map(center=[34.0, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:maxM Function

Returns the maximum measurement coordinate of a given geometry.

In [14]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?maxM
WHERE {
  my:PExactGeom geo:asWKT ?literal .
  BIND(geof:maxM(?literal) AS ?maxM)
}
"""),["literal"],None)

Variable,Value
literal,"LineString M (-83.4 34.0 5, -83.3 34.3 10)"
maxM,10.0


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:minM Function

Returns the minimum measurement coordinate of a geometry.

In [15]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?minM
WHERE {
  my:PExactGeom geo:asWKT ?literal .
  BIND(geof:minM(?literal) AS ?minM)
}
"""),["literal"],None)

Variable,Value
literal,"LineString M (-83.4 34.0 5, -83.3 34.3 10)"
minM,5.0


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:startPoint Function

Returns the first point of a given geometry in the CRS and literal format of the input geometry.

In [16]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?startPoint
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:startPoint(?literal) AS ?startPoint)
}
"""),["literal","startPoint"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
startPoint,POINT (-83.6 34.1)


Map(center=[34.1, -83.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:X Function

Returns the X coordinate of a point geometry. The X axis is determined from its CRS definition.

In [17]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?x
WHERE {
  my:FExactGeom geo:asWKT ?literal .
  BIND(geof:X(?literal) AS ?x)
}
"""),["literal"],None)

Variable,Value
literal,Point(-83.4 34.4)
x,-83.4


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:Y Function

Returns the Y coordinate of a Point geometry. The Y axis is determined by its CRS definition.

In [18]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?y
WHERE {
  my:FExactGeom geo:asWKT ?literal .
  BIND(geof:Y(?literal) AS ?y)
}
"""),["literal"],None)

Variable,Value
literal,Point(-83.4 34.4)
y,34.4


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:Z Function

In [19]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?z
WHERE {
  my:NPointGeom geo:asWKT ?literal .
  BIND(geof:Z(?literal) AS ?z)
}
"""),["literal"],None)

Variable,Value
literal,Point Z (-77.059 38.9 1)
z,1.0


Map(center=[38.9, -77.059], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## Non-topological query functions

### geofext:azimuth Function

In [20]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?azimuth
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:azimuth(?aLiteral) as ?azimuth)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
azimuth,90.0


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:boundingDiagonal Function

Returns the diagonal of a geometrie's bounding box, i.e. a 2 point LineString from its minimum to maximum coordinates.

In [21]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bdiag
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:boundingDiagonal(?aLiteral) as ?bdiag)
}
"""),["aLiteral","bdiag"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bdiag,"LINESTRING (-83.6 34.1, -83.2 34.5)"


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:closestPoint Function

Returns the closest point between two input geometries. This point is the first point of the shortest line between the geometries.

In [22]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?cPoint
WHERE {
  my:A my:hasPointGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:closestPoint(?aLiteral, ?dLiteral) as ?cPoint)
}
"""),["aLiteral","dLiteral","cPoint"],None)

UnboundLocalError: cannot access local variable 'resultlist' where it is not associated with a value

### geofext:compactnessRatio Function

Calculates the compactness ratio of a polygon, an indicator of polygon shape complexity.

In [ ]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cratio
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:compactnessRatio(?aLiteral) as ?cratio)
}
"""),["aLiteral"],None)

### geofext:exteriorRing Function

In [ ]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?exRing
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:exteriorRing(?literal) AS ?exRing)
}
"""),["literal","exRing"],None)

### geofext:farthestCoordinate Function

In [ ]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?farthestCoord
WHERE {
  my:A my:hasPointGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:farthestCoordinate(?aLiteral, ?dLiteral) as ?farthestCoord)
}
"""),["aLiteral","dLiteral","farthestCoord"],None)

### geofext:frechetDistance Function

In [ ]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?fdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:frechetDistance(?cLiteral, ?fLiteral) as ?fdistance)
}
"""),["cLiteral","fLiteral"],None)

### geofext:force2D Function

Creates a 2D geometry from a given geometry of a higher dimension by reducing its dimensionality

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?force2D
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:force2D(?literal) AS ?force2D)
}
"""),["literal","force2D"],None)

### geofext:force3D Function

Creates a 3D geometry by adding a given Z coordinate to every point of the input geometry

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?force3D
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:force3D(?literal,"5.0"^^xsd:double) AS ?force3D)
}
"""),["literal","force3D"],None)

### geofext:hausdorffDistance Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?hdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:hausdorffDistance(?cLiteral, ?fLiteral) as ?hdistance)
}
"""),["cLiteral","fLiteral"],None)

### geofext:isClosed Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isAClosed ?isEClosed {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isClosed(?a_wkt) AS ?isAClosed)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isClosed(?e_wkt) AS ?isEClosed)
}
"""),["a_wkt","e_wkt"],None)

### geofext:isCollection Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?C1 ?C2 ?C3 ?isC1 ?isC2 ?isC3 {
    BIND("GEOMETRYCOLLECTION (MULTIPOINT((0 0), (1 1)), POINT(3 4), LINESTRING(2 3, 3 4))"^^geo:wktLiteral as ?C1)
    BIND("MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"^^geo:wktLiteral as ?C2)
    BIND("POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0), (1 1, 1 2, 2 2, 2 1, 1 1))"^^geo:wktLiteral as ?C3)
    BIND(geof:isCollection(?C1) AS ?isC1)
    BIND(geof:isCollection(?C2) AS ?isC2)
    BIND(geof:isCollection(?C3) AS ?isC3)
}
"""),["C1","C2","C3"],None)

### geofext:isCCW Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isACW ?isECW {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isCCW(?a_wkt) AS ?isACW)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isCCW(?e_wkt) AS ?isECW)
}
"""),["a_wkt","e_wkt"],None)

### geofext:isCW Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isACW ?isECW {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isCW(?a_wkt) AS ?isACW)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isCW(?e_wkt) AS ?isECW)
}
"""),["a_wkt","e_wkt"],None)

### geofext:isRectangle Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?isRectangle ?isNoRectangle {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isRectangle(?a_wkt) AS ?isNoRectangle)
    BIND(geof:isRectangle(geofo:envelope(?a_wkt)) AS ?isRectangle)
}
"""),["a_wkt"],None)

### geofext:isTriangle Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?isTriangle ?isNoTriangle {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isTriangle(?a_wkt) AS ?isNoTriangle)
    BIND(geof:isTriangle("POLYGON ((0 0,0 1,1 1,0 0))"^^geo:wktLiteral) AS ?isTriangle)
}
"""),["a_wkt"],None)

### geofext:isRing Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?a_wkt ?o_wkt ?isARing ?isORing {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isRing(?a_wkt) AS ?isARing)
            <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
    BIND(geof:isRing("POINT M (1 2 3)"^^geo:wktLiteral) AS ?isORing)
}
"""),["a_wkt","o_wkt"],None)

### geofext:isValid Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?o_wkt ?a_isValid ?o_isNotValid {
<http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
BIND(geof:isValid(?a_wkt) AS ?a_isValid)
        <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
BIND(geof:isValid("POLYGON((0 0, 10 10, 0 10, 10 0, 0 0))"^^geo:wktLiteral) AS ?o_isNotValid)
}
"""),["a_wkt","o_wkt"],None)

### geofext:isValidTrajectory Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?o_wkt ?validTGeom ?isValidT ?O_isValidT2 ?A_isValidT {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isValidTrajectory(?a_wkt) AS ?A_isValidT)
    <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
    BIND(geof:isValidTrajectory(?o_wkt) AS ?O_isValidT2)
    BIND("LineString M(0 0 1, 10 10 2, 0 10 3, 10 0 4, 0 0 5)"^^geo:wktLiteral AS ?validTGeom)
    BIND(geof:isValidTrajectory(?validTGeom) AS ?isValidT)
}
"""),["a_wkt","o_wkt","validTGeom"],None)

### geofext:longestLine Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?aLiteral ?dLiteral ?longestLine
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:longestLine(?aLiteral, ?dLiteral) as ?longestLine)
}
"""),["aLiteral","dLiteral","longestLine"],None)

### geofext:maxDistance Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?aLiteral ?dLiteral ?maxDistance
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:maxDistance(?aLiteral, ?dLiteral) as ?maxDistance)
}
"""),["aLiteral","dLiteral"],None)

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?maxM
WHERE {
  my:PExactGeom geo:asWKT ?literal .
  BIND(geof:maxM(?literal) AS ?maxM)
}
"""),["aLiteral","dLiteral"],None)

### geofext:metricWithinDistance Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral (xsd:boolean(?wdistance) as ?wddistance)
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:metricWithinDistance(?cLiteral, ?fLiteral,"5.0"^^xsd:double) as ?wdistance)
}
"""),["cLiteral","fLiteral"],None)

### geofext:minimumBoundingRadius Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bradius
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumBoundingRadius(?aLiteral) as ?bradius)
}
"""),["aLiteral"],None)

### geofext:minimumClearance Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?mclearance
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumClearance(?aLiteral) as ?mclearance)
}
"""),["aLiteral"],None)

### geofext:minimumClearanceLine Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?mclearancel
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumClearanceLine(?aLiteral) as ?mclearancel)
}
"""),["aLiteral","mclearancel"],None)

### geofext:numInteriorRing Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numInteriorRing
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numInteriorRing(?literal) AS ?numInteriorRing)
}
"""),["literal"],None)

### geofext:numPatches Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numPatches
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numPatches(?literal) AS ?numPatches)
}
"""),["literal"],None)

### geofext:numPoints Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numPoints
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numPoints(?literal) AS ?numPoints)
}
"""),["literal"],None)

### geofext:offsetCurve Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?offsetCurve
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:offsetCurve(?literal,10) AS ?offsetCurve)
}
"""),["literal"],None)

### geofext:pointN Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?pointN
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:pointN(?literal,"1"^^xsd:integer) AS ?pointN)
}
"""),["literal"],None)

### geofext:pointOnSurface Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?pointOnSurface
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:pointOnSurface(?literal) AS ?pointOnSurface)
}
"""),["literal"],None)

### geofext:removeRepeatedPoints Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal  ?rep
WHERE {
  BIND("POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"^^geo:wktLiteral AS ?literal)
  BIND(geof:removeRepeatedPoints(?literal) AS ?rep)
}
"""),["literal"],None)

### geofext:reverse Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?reverse
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:reverse(?literal) AS ?reverse)
}
"""),["literal"],None)

### geofext:scale Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?scale
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:scale(?literal,"2.0"^^xsd:double,"2.0"^^xsd:double,"2.0"^^xsd:double) AS ?scale)
}
"""),["literal"],None)

### geofext:sharedPaths Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?literal2 ?spaths
WHERE {
  my:EExactGeom geo:asWKT ?literal .
  my:EExactGeom geo:asWKT ?literal2 .
  BIND(geof:sharedPaths(?literal,?literal2) AS ?spaths)
}
"""),["literal","literal2","spaths"],None)

### geofext:shortestLine Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?literal2 ?sline
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  my:OExactGeom geo:asWKT ?literal2 .
  BIND(geof:shortestLine(?literal,?literal2) AS ?sline)
}
"""),["literal","literal2","sline"],None)

### geofext:simplify Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?simple
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:simplify(?literal,"1.5"^^xsd:double) AS ?simple)
}
"""),["literal","simple"],None)

### geofext:translate Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?translate
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:translate(?literal,"1.0"^^xsd:double,"1.0"^^xsd:double,"1.0"^^xsd:double) AS ?translate)
}
"""),["literal","translate"],None)

### geofext:transformCRS84 Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?transformed
WHERE {
  my:LExactGeom geo:asWKT ?literal .
  BIND(geof:transformCRS84(?literal) AS ?transformed)
}
"""),["literal","transformed"],None)

### geofext:withinDistance Function

In [ ]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?wdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:withinDistance(?cLiteral, ?fLiteral,"5.0"^^xsd:double) as ?wdistance)
}
"""),["cLiteral","fLiteral"],None)